# 01. 데이터 적재·품질 확인 (Day 1)
**목적**: CSV 9개를 DuckDB에 등록하고, 행 수·결측·분석 모집단·지연율·저평점률을 계획서(부록 B) 수치와 대조해 내 SQL이 맞는지 검증한다.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("data/interim/olist.duckdb")   # 파일로 저장됨
RAW = "data/raw/"

In [2]:
def load(name, fname, types=None):
    opt = f", types={types}" if types else ""
    con.execute(f"CREATE OR REPLACE VIEW {name} AS "
                f"SELECT * FROM read_csv('{RAW}{fname}', header=true{opt})")

load("orders",         "olist_orders_dataset.csv")
load("order_items",    "olist_order_items_dataset.csv")
load("order_reviews",  "olist_order_reviews_dataset.csv")
load("order_payments", "olist_order_payments_dataset.csv")
load("products",       "olist_products_dataset.csv")
load("cat_tr",         "product_category_name_translation.csv")
# 우편번호는 파일에 따옴표가 붙어 있어 문자열로 읽힘 → 정수로 지정 (조인 키 타입 통일)
load("customers",   "olist_customers_dataset.csv",   {"customer_zip_code_prefix": "INTEGER"})
load("sellers",     "olist_sellers_dataset.csv",     {"seller_zip_code_prefix": "INTEGER"})
load("geolocation", "olist_geolocation_dataset.csv", {"geolocation_zip_code_prefix": "INTEGER"})

In [3]:
# 참고: rev·chk는 아래 셀에서 만든 테이블이라 재실행하면 목록에 함께 보임
print(con.execute("SHOW TABLES").df().name.tolist())

for t, c in [("customers","customer_zip_code_prefix"),
             ("sellers","seller_zip_code_prefix"),
             ("geolocation","geolocation_zip_code_prefix")]:
    print(t, con.execute(f"SELECT typeof({c}) FROM {t} LIMIT 1").fetchone()[0])

['base', 'cat_tr', 'chk', 'customers', 'geo_clean', 'geo_rep', 'geolocation', 'item_base', 'item_lvl', 'master', 'ord_agg', 'ord_pay', 'ord_sum', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'rep_dist', 'rep_price', 'rev', 'sellers', 'zip_status']
customers INTEGER
sellers INTEGER
geolocation INTEGER


In [4]:
tables = ["orders","order_items","order_reviews","order_payments",
          "customers","sellers","products","geolocation","cat_tr"]
sql = " UNION ALL ".join(f"SELECT '{t}' AS tbl, COUNT(*) AS n_rows FROM {t}" for t in tables)
con.execute(sql).df()

,tbl,n_rows
0,orders,99441
1,order_items,112650
2,order_reviews,99224
3,order_payments,103886
4,customers,99441
5,sellers,3095
6,products,32951
7,geolocation,1000163
8,cat_tr,71


In [5]:
# [PK 유일성 검증] Day1 재검토(2026-09-22)에서 지적된 항목 — 행 수만 보고 PK 유일성은 확인 안 했던 부분
# orders/customers/sellers/products/order_items(복합키)/cat_tr 전부 유일해야 함
pk_check = con.execute("""
WITH oi_dedup AS (
    SELECT DISTINCT order_id, order_item_id FROM order_items
)
SELECT
    (SELECT COUNT(*) - COUNT(DISTINCT order_id)      FROM orders)    AS orders_dup,
    (SELECT COUNT(*) - COUNT(DISTINCT customer_id)    FROM customers) AS customers_dup,
    (SELECT COUNT(*) - COUNT(DISTINCT seller_id)      FROM sellers)   AS sellers_dup,
    (SELECT COUNT(*) - COUNT(DISTINCT product_id)     FROM products)  AS products_dup,
    (SELECT COUNT(*) FROM order_items) - (SELECT COUNT(*) FROM oi_dedup) AS order_items_pk_dup,
    (SELECT COUNT(*) - COUNT(DISTINCT product_category_name) FROM cat_tr) AS cat_tr_dup
""").df()
assert (pk_check.values == 0).all(), f"PK 유일성 위반 발견:\n{pk_check}"
pk_check


,orders_dup,customers_dup,sellers_dup,products_dup,order_items_pk_dup,cat_tr_dup
0,0,0,0,0,0,0


In [6]:
con.execute("DESCRIBE orders").df()[["column_name", "column_type"]]

,column_name,column_type
0,order_id,VARCHAR
1,customer_id,VARCHAR
2,order_status,VARCHAR
3,order_purchase_timestamp,TIMESTAMP
4,order_approved_at,TIMESTAMP
5,order_delivered_carrier_date,TIMESTAMP
6,order_delivered_customer_date,TIMESTAMP
7,order_estimated_delivery_date,TIMESTAMP


In [7]:
con.execute("""
SELECT COUNT(*)                                   AS n_rows,
       COUNT(*) - COUNT(review_comment_title)     AS n_title_null,
       COUNT(*) - COUNT(review_comment_message)   AS n_msg_null,
       ROUND(100.0 * (COUNT(*) - COUNT(review_comment_message)) / COUNT(*), 1) AS pct_msg_null
FROM order_reviews
""").df()

,n_rows,n_title_null,n_msg_null,pct_msg_null
0,99224,87656,58247,58.7


In [8]:
def null_profile(table):
    cols = con.execute(f"SELECT column_name FROM (DESCRIBE SELECT * FROM {table})").df().column_name
    sql = " UNION ALL ".join(
        f'SELECT \'{c}\' AS col, COUNT(*) - COUNT("{c}") AS n_null, COUNT(*) AS n_rows FROM {table}'
        for c in cols)
    df = con.execute(sql).df()
    df["pct_null"] = (df.n_null / df.n_rows * 100).round(1)
    return df[df.n_null > 0][["col", "n_null", "pct_null"]]

for t in ["orders","order_items","order_reviews","order_payments",
          "customers","sellers","products","geolocation","cat_tr"]:
    res = null_profile(t)
    print(f"\n[{t}]")
    print(res.to_string(index=False) if len(res) else "결측 없음")


[orders]
                          col  n_null  pct_null
            order_approved_at     160       0.2
 order_delivered_carrier_date    1783       1.8
order_delivered_customer_date    2965       3.0

[order_items]
결측 없음

[order_reviews]
                   col  n_null  pct_null
  review_comment_title   87656      88.3
review_comment_message   58247      58.7

[order_payments]
결측 없음

[customers]
결측 없음

[sellers]
결측 없음

[products]
                       col  n_null  pct_null
     product_category_name     610       1.9
       product_name_lenght     610       1.9
product_description_lenght     610       1.9
        product_photos_qty     610       1.9
          product_weight_g       2       0.0
         product_length_cm       2       0.0
         product_height_cm       2       0.0
          product_width_cm       2       0.0

[geolocation]
결측 없음

[cat_tr]
결측 없음


In [9]:
con.execute("""
SELECT order_status,
       COUNT(*)                                        AS n_orders,
       COUNT(order_delivered_customer_date)            AS n_has_date,
       COUNT(*) - COUNT(order_delivered_customer_date) AS n_null_date
FROM orders
GROUP BY order_status
ORDER BY n_orders DESC
""").df()

,order_status,n_orders,n_has_date,n_null_date
0,delivered,96478,96470,8
1,shipped,1107,0,1107
2,canceled,625,6,619
3,unavailable,609,0,609
4,invoiced,314,0,314
5,processing,301,0,301
6,created,5,0,5
7,approved,2,0,2


In [10]:
con.execute("""
CREATE OR REPLACE TABLE rev AS
SELECT * EXCLUDE rn FROM (
  SELECT *, ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id ASC) AS rn
  FROM order_reviews)
WHERE rn = 1
""")
con.execute("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM rev").df()

,n_rows,n_orders
0,98673,98673


In [11]:
con.execute("""
SELECT COUNT(*)                                        AS n_window,
       COUNT(order_delivered_customer_date)            AS n_delivered,
       ROUND(100.0 * SUM(CASE WHEN CAST(order_delivered_customer_date AS DATE)
                                  > CAST(order_estimated_delivery_date AS DATE)
                              THEN 1 ELSE 0 END)
             / COUNT(order_delivered_customer_date), 2) AS late_pct
FROM orders
WHERE order_purchase_timestamp >= '2017-01-01'
  AND order_purchase_timestamp <  '2018-09-01'
""").df()

,n_window,n_delivered,late_pct
0,99092,96204,6.79


In [12]:
# [2단계 확인용 임시 테이블] 리뷰가 있는 주문만 남김 (INNER JOIN) → 95,561건
# 리뷰 없는 주문 643건은 여기서 빠짐. 1단계 모집단(96,204건)이 아님!
# 최종 마스터 테이블은 LEFT JOIN + low NULL로 따로 만든다 (Day 3)
con.execute("""
CREATE OR REPLACE TABLE chk AS
SELECT o.order_id,
       CASE WHEN CAST(o.order_delivered_customer_date AS DATE)
                 > CAST(o.order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END AS late,
       r.review_score,
       CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END AS low,
       r.review_comment_message
FROM orders o JOIN rev r USING (order_id)
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
""")

# (1) 전체 비율
display(con.execute("""
SELECT COUNT(*) AS n_orders,
       ROUND(100.0 * AVG(low), 1) AS low_pct,
       ROUND(100.0 * COUNT(review_comment_message) / COUNT(*), 1) AS comment_pct
FROM chk""").df())

# (2) 지연 × 저평점
display(con.execute("""
SELECT late, COUNT(*) AS n, SUM(low) AS n_low, ROUND(100.0 * AVG(low), 1) AS low_pct
FROM chk GROUP BY late ORDER BY late""").df())

# (3) 3단계 대상: 지연 아님 & 저평점
display(con.execute("""
SELECT COUNT(*) AS n_stage3, COUNT(review_comment_message) AS n_with_comment
FROM chk WHERE late = 0 AND low = 1""").df())

# (4) 멀티셀러 주문 — 주의: order_items 전체(기간·배송 필터 없음) 기준 1,278건(1.3%)
#     분석 모집단(96,204건) 기준은 1,272건(1.32%). D-05는 후자로 계산
display(con.execute("""
SELECT COUNT(*) AS n_orders_with_items,
       SUM(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END) AS n_multi,
       ROUND(100.0 * AVG(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END), 2) AS multi_pct
FROM (SELECT order_id, COUNT(DISTINCT seller_id) AS n_sellers
      FROM order_items GROUP BY order_id)""").df())

# 배송완료 주문 중 리뷰가 없는 건수 (chk가 INNER JOIN으로 빠뜨리는 주문)
con.execute("""
SELECT COUNT(*) AS n_delivered_window,
       COUNT(r.order_id) AS n_with_review,
       COUNT(*) - COUNT(r.order_id) AS n_no_review
FROM orders o LEFT JOIN rev r USING (order_id)
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
""").df()

,n_orders,low_pct,comment_pct
0,95561,12.8,40.5


,late,n,n_low,low_pct
0,0,89182,8247.0,9.2
1,1,6379,3981.0,62.4


,n_stage3,n_with_comment
0,8247,6449


,n_orders_with_items,n_multi,multi_pct
0,98666,1278.0,1.3


,n_delivered_window,n_with_review,n_no_review
0,96204,95561,643


## 코드 리뷰 지적 사항 검증
`decisions.md`에 적은 숫자들이 내 SQL로 재현되는지 확인한다. (멀티셀러 분석 모집단 기준, 분석기간 내 delivered·canceled, 리뷰 중복, 기타, 결정 문서의 나머지 사실)

In [13]:
# [검증 1] 멀티셀러: 분석 모집단(96,204건) 기준 → 기대: 96,204 / 1,272 / 1.32
con.execute("""
SELECT COUNT(*) AS n_pop_with_items,
       SUM(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END) AS n_multi,
       ROUND(100.0 * AVG(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END), 2) AS multi_pct
FROM (SELECT order_id, COUNT(DISTINCT seller_id) AS n_sellers
      FROM order_items GROUP BY order_id) s
JOIN (SELECT order_id FROM orders
      WHERE order_delivered_customer_date IS NOT NULL
        AND order_purchase_timestamp >= '2017-01-01'
        AND order_purchase_timestamp <  '2018-09-01') p
USING (order_id)
""").df()

,n_pop_with_items,n_multi,multi_pct
0,96204,1272.0,1.32


In [14]:
# [검증 2] 분석기간 내 delivered/canceled의 도착일 유무
# 기대: canceled 580(도착일 있음 1, 없음 579) / delivered 96,211(있음 96,203, 없음 8)
# → 96,203 + 1 = 96,204 = 분석 모집단
con.execute("""
SELECT order_status,
       COUNT(*) AS in_window,
       COUNT(order_delivered_customer_date) AS has_date,
       COUNT(*) - COUNT(order_delivered_customer_date) AS no_date
FROM orders
WHERE order_status IN ('delivered', 'canceled')
  AND order_purchase_timestamp >= '2017-01-01'
  AND order_purchase_timestamp <  '2018-09-01'
GROUP BY order_status ORDER BY order_status
""").df()

,order_status,in_window,has_date,no_date
0,canceled,580,1,579
1,delivered,96211,96203,8


In [15]:
# [검증 3] 리뷰 중복 두 종류
# (a) order_id 기준 → 기대: 547 / 551
display(con.execute("""
SELECT COUNT(*) AS n_orders_multi_review, SUM(c - 1) AS extra_rows
FROM (SELECT order_id, COUNT(*) AS c FROM order_reviews GROUP BY order_id HAVING COUNT(*) > 1)
""").df())

# (b) review_id 기준(원본) → 기대: 789 / 764 / 25 / 814
display(con.execute("""
SELECT COUNT(*) AS n_ids,
       SUM(CASE WHEN c = 2 THEN 1 ELSE 0 END) AS in_2_orders,
       SUM(CASE WHEN c = 3 THEN 1 ELSE 0 END) AS in_3_orders,
       SUM(c - 1) AS extra_rows
FROM (SELECT review_id, COUNT(*) AS c FROM order_reviews GROUP BY review_id HAVING COUNT(*) > 1)
""").df())

# (c) 주문 단위 dedup(rev) 후에도 남는 review_id 중복 → 기대: 561 / 1,139
display(con.execute("""
SELECT COUNT(*) AS n_ids, SUM(c) AS n_orders
FROM (SELECT review_id, COUNT(*) AS c FROM rev GROUP BY review_id HAVING COUNT(*) > 1)
""").df())

# (d) 3단계 대상 코멘트: 주문 수 vs 고유 review_id → 기대: 6,449 / 6,400
display(con.execute("""
SELECT COUNT(r.review_comment_message) AS n_comment_orders,
       COUNT(DISTINCT CASE WHEN r.review_comment_message IS NOT NULL THEN r.review_id END) AS n_distinct_review_id
FROM chk c JOIN rev r USING (order_id)
WHERE c.late = 0 AND c.low = 1
""").df())

,n_orders_multi_review,extra_rows
0,547,551.0


,n_ids,in_2_orders,in_3_orders,extra_rows
0,789,764.0,25.0,814.0


,n_ids,n_orders
0,561,1139.0


,n_comment_orders,n_distinct_review_id
0,6449,6400


In [16]:
# [검증 4] 기타
# (a) 결제 회차 0값 → 기대: credit_card 2건
display(con.execute("""
SELECT payment_type, COUNT(*) AS n FROM order_payments
WHERE payment_installments = 0 GROUP BY payment_type""").df())

# (b) 우편번호 원본 문자열의 앞자리 0 → 기대: sellers 3,095행 중 1,027 / customers 99,441행 중 23,995 (전부 5자리)
for t, f, c in [("sellers",   "olist_sellers_dataset.csv",   "seller_zip_code_prefix"),
                ("customers", "olist_customers_dataset.csv", "customer_zip_code_prefix")]:
    print(t, con.execute(f"""
        SELECT COUNT(*) AS n_rows,
               SUM(CASE WHEN {c} LIKE '0%' THEN 1 ELSE 0 END) AS n_lead0,
               MIN(LENGTH({c})) AS len_min, MAX(LENGTH({c})) AS len_max
        FROM read_csv('{RAW}{f}', header=true, all_varchar=true)""").fetchone())

# (c) DuckDB DESC 정렬에서 NULL 위치 → 기대: 3, 1, 마지막 행이 NULL(<NA> 등으로 표시) = NULLS LAST
display(con.execute("SELECT x FROM (VALUES (1), (NULL), (3)) t(x) ORDER BY x DESC").df())

,payment_type,n
0,credit_card,2


sellers (3095, 1027, 5, 5)
customers (99441, 23995, 5, 5)


,x
0,3
1,1
2,<NA>


In [17]:
# [검증 5] decisions.md에 적은 나머지 사실들
# (a) 같은 review_id로 묶인 행들의 내용이 같은가 → 기대: 789 / 789 / 789 (전부 동일)
display(con.execute("""
SELECT COUNT(*) AS n_dup_ids,
       SUM(CASE WHEN n_score = 1 AND n_title = 1 AND n_msg = 1 THEN 1 ELSE 0 END) AS same_score_title_msg,
       SUM(CASE WHEN n_created = 1 AND n_answer = 1 THEN 1 ELSE 0 END) AS same_timestamps
FROM (SELECT review_id,
             COUNT(DISTINCT review_score) AS n_score,
             COUNT(DISTINCT COALESCE(review_comment_title, '<NULL>')) AS n_title,
             COUNT(DISTINCT COALESCE(review_comment_message, '<NULL>')) AS n_msg,
             COUNT(DISTINCT review_creation_date) AS n_created,
             COUNT(DISTINCT review_answer_timestamp) AS n_answer
      FROM order_reviews GROUP BY review_id HAVING COUNT(*) > 1)
""").df())

# (b) 분석기간 내 canceled + 도착일 있는 주문 → 기대: 1950d777... / 2018-02-19 / canceled / 3
# late도 같이 계산 (Day1 재검토: 이 1건이 late=1인지 미기록이었음 → decisions.md D-00에 반영)
display(con.execute("""
SELECT o.order_id, CAST(o.order_purchase_timestamp AS DATE) AS purchase_date,
       o.order_status, r.review_score,
       CASE WHEN CAST(o.order_delivered_customer_date AS DATE)
                 > CAST(o.order_estimated_delivery_date AS DATE)
            THEN 1 ELSE 0 END AS late
FROM orders o LEFT JOIN order_reviews r USING (order_id)
WHERE o.order_status = 'canceled' AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01' AND o.order_purchase_timestamp < '2018-09-01'
""").df())

# (c) installments 0 주문의 결제 행 수 → 기대: 주문 2개, 각각 1행, installments_max 0
display(con.execute("""
SELECT order_id, COUNT(*) AS n_payment_rows, MAX(payment_installments) AS installments_max
FROM order_payments
WHERE order_id IN (SELECT order_id FROM order_payments WHERE payment_installments = 0)
GROUP BY order_id
""").df())

,n_dup_ids,same_score_title_msg,same_timestamps
0,789,789.0,789.0


,order_id,purchase_date,order_status,review_score,late
0,1950d777989f6a877539f53795b4c3c3,2018-02-19,canceled,3,1


,order_id,n_payment_rows,installments_max
0,744bade1fcf9ff3f31d860ace076d422,1,0
1,1a57108394169c0b47d8f876acc9ba2d,1,0


## 결론
- 행 수·결측 모두 계획서와 일치 (geolocation 1,000,163행, 결측 없음)
- 분석 모집단(도착일이 있는 분석기간 내 주문) 96,204건, 지연율 6.79% 재현 / 리뷰 있는 주문 95,561건 중 저평점 12.8%
- 지연 주문의 저평점률 62.4% vs 비지연 9.2%. 저평점 주문의 약 67.4%(8,247건)는 비지연 주문에서 발생 → 3단계 필요
  (2026-09-22 재검토: 이 62.4%/9.2% 비교는 리뷰 응답 시점을 통제 안 한 값. 도착 후 응답만 보면 19.5%/9.2%=2.1배로 완만해짐. decisions.md D-00c 참고)
- 코드 리뷰에서 나온 지적 사항의 숫자(멀티셀러 1,272 / 리뷰 중복 547·789·561 / 취소 1건 등)도 SQL로 재현해 확인
- PK 유일성(orders/customers/sellers/products/order_items 복합키/cat_tr)도 명시적으로 assert (2026-09-22 추가)


In [18]:
con.close()